# 02 — Train: Multi-task parcel detection (v2.0, Colab)

**Runtime: GPU (T4 minimum; L4 / A100 faster).** Runtime menu → Change runtime type → GPU.

Prereq: `01_prep_colab.ipynb` has run and `/content/drive/MyDrive/aigeolab_train/{tiles, labels, manifest.csv}` exists.

**What changed in v2.0:** the model is now **UNet++ with an EfficientNet-B3 encoder** and predicts **three heads** instead of one:
1. **Boundary** — the fence-line (same as v1).
2. **Interior** — filled plot body, eroded inward to avoid the boundary band. Used at inference to get clean connected components even when the boundary is broken.
3. **Corner heatmap** — gaussian peaks at every polygon vertex. Used at inference to snap rect/trapezoid vertices to true junctions.

Pipeline:
1. Install deps + mount Drive + clone repo.
2. Load config; resolve Colab paths.
3. Read manifest; mouza-disjoint train/val split.
4. Rasterise **3 masks per tile** (boundary, interior, corners) + a 'valid' coverage mask.
5. Patch enumeration; drop patches that don't overlap annotated coverage.
6. Copy tiles + masks to Colab's local SSD.
7. Pre-extract all patches into RAM (zero file I/O at training time).
8. In-memory Dataset returning `(img, boundary, interior, corners, valid)`.
9. Train UNet++ with valid-masked multi-task BCE+Dice loss, per-head val IoU.
10. Qualitative eval overlay (boundary / interior / corners / final polygons).

In [ ]:
# --- Cell 1: install deps + mount Drive + clone repo ---
!pip install -q rasterio shapely segmentation-models-pytorch albumentations opencv-python-headless pyyaml pyshp

from google.colab import drive
drive.mount('/content/drive')

import os, subprocess
REPO_URL = 'https://github.com/tahmid013/AIGEOLAB_OFFICE.git'
REPO_DIR = '/content/AIGEOLAB_OFFICE'
if os.path.isdir(REPO_DIR):
    print(subprocess.run(['git', '-C', REPO_DIR, 'pull'], capture_output=True, text=True).stdout)
else:
    print(subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], capture_output=True, text=True).stdout)
os.chdir(REPO_DIR)


In [ ]:
# --- Cell 2: load config, resolve Colab paths ---
import yaml
from pathlib import Path

with open('config.yaml') as f: CFG = yaml.safe_load(f)
ENV = 'colab'
P = CFG['paths'][ENV]
STAGE = Path(P['staging_root'])
assert STAGE.is_dir(), f'{STAGE} not found. Run 01_prep_colab.ipynb first.'
TILES   = STAGE / 'tiles'
LABELS  = STAGE / 'labels'
MASKS   = STAGE / 'masks';   MASKS.mkdir(exist_ok=True)
print('staging:', STAGE)
print('manifest exists:', (STAGE / 'manifest.csv').exists())
print('tiles  :', len(list(TILES.glob('*.tif'))))
print('labels :', len(list(LABELS.glob('*.shp'))))


In [ ]:
# --- Cell 3: read manifest, mouza-disjoint train/val split ---
import csv, random
rows = []
with open(STAGE / 'manifest.csv', newline='', encoding='utf-8') as fh:
    for r in csv.DictReader(fh):
        rows.append({'x': int(r['tile_x_km']), 'y': int(r['tile_y_km']),
                     'tif': STAGE / r['tile_tif_relpath'],
                     'mouzas': r['mouza_label_stems'].split('|')})
print(f'manifest rows: {len(rows)}')

all_mouzas = sorted({m for r in rows for m in r['mouzas']})
random.Random(CFG['split']['seed']).shuffle(all_mouzas)
n_val = max(1, int(len(all_mouzas) * CFG['split']['val_mouza_fraction']))
if len(all_mouzas) - n_val < 1:
    n_val = len(all_mouzas) - 1
val_mouzas   = set(all_mouzas[:n_val])
train_mouzas = set(all_mouzas[n_val:])
print(f'Train mouzas: {len(train_mouzas)}  Val mouzas: {len(val_mouzas)}')
print('  val =', sorted(val_mouzas))


In [ ]:
# --- Cell 4: rasterise boundary + interior + corners + valid masks (cached) ---
# boundary : 3-px lines along every plot polygon (head 0)
# interior : filled polygons, eroded inward by `interior_erode_px` so they don't
#            overlap the boundary band (head 1)
# corners  : gaussian peaks at every polygon vertex/junction (head 2)
# valid    : where labels are meaningful (filled polygons + dilation). The model
#            isn't rewarded or penalised outside this region.
import numpy as np, rasterio, cv2, shapefile
from tqdm.auto import tqdm

THICK         = CFG['dataset']['boundary_thickness_px']
INTERIOR_ERODE = CFG['dataset'].get('interior_erode_px', max(1, THICK))
CORNER_SIGMA   = CFG['dataset'].get('corner_sigma_px', 4)
VALID_PAD = max(8, THICK + 5)

def read_polys_pyshp(shp_path):
    rdr = shapefile.Reader(str(shp_path))
    polys = []
    for shp in rdr.shapes():
        pts = shp.points
        parts = list(shp.parts) + [len(pts)]
        for i in range(len(parts) - 1):
            polys.append(pts[parts[i]: parts[i + 1]])
    return polys


def _gaussian_splat(canvas: np.ndarray, x: int, y: int, sigma: float):
    """Add a normalised gaussian peak at (x, y) to a float32 canvas (in-place, max)."""
    H, W = canvas.shape
    rad = int(max(3, 3 * sigma))
    x0, x1 = max(0, x - rad), min(W, x + rad + 1)
    y0, y1 = max(0, y - rad), min(H, y + rad + 1)
    if x1 <= x0 or y1 <= y0: return
    yy, xx = np.mgrid[y0:y1, x0:x1]
    g = np.exp(-((xx - x) ** 2 + (yy - y) ** 2) / (2 * sigma * sigma)).astype(np.float32)
    canvas[y0:y1, x0:x1] = np.maximum(canvas[y0:y1, x0:x1], g)


def heads_and_valid(tif_path, mouza_stems,
                    thick=THICK, erode=INTERIOR_ERODE, sigma=CORNER_SIGMA, pad=VALID_PAD):
    with rasterio.open(tif_path) as ds:
        H, W = ds.height, ds.width
        T = ds.transform
    boundary = np.zeros((H, W), dtype=np.uint8)
    filled   = np.zeros((H, W), dtype=np.uint8)
    corners  = np.zeros((H, W), dtype=np.float32)

    for stem in mouza_stems:
        shp_path = LABELS / f'{stem}.shp'
        if not shp_path.exists():
            print(f'  WARN missing label: {shp_path.name}'); continue
        for ring in read_polys_pyshp(shp_path):
            xs = np.array([p[0] for p in ring], dtype=np.float64)
            ys = np.array([p[1] for p in ring], dtype=np.float64)
            cols, rows_ = ~T * (xs, ys)
            pts = np.stack([cols, rows_], axis=1).astype(np.int32)
            cv2.polylines(boundary, [pts.reshape(-1, 1, 2)], isClosed=True, color=255, thickness=thick)
            cv2.fillPoly(filled, [pts], color=255)
            for x, y in pts:
                if 0 <= x < W and 0 <= y < H:
                    _gaussian_splat(corners, int(x), int(y), sigma)

    # Interior = filled, eroded inward so head 1 doesn't overlap head 0.
    k_erode = np.ones((erode, erode), dtype=np.uint8)
    interior = cv2.erode(filled, k_erode)

    # Valid coverage = filled polygons dilated outward; covers boundary band too.
    k_dilate = np.ones((pad, pad), dtype=np.uint8)
    valid = cv2.dilate(filled, k_dilate)

    # Corners stored as uint8 [0,255] for cheap PNG caching.
    corners_u8 = (corners.clip(0.0, 1.0) * 255.0).astype(np.uint8)
    return boundary, interior, corners_u8, valid


for r in tqdm(rows, desc='rasterising (v2.0: 3 heads + valid)'):
    stem = Path(r['tif']).stem
    bp = MASKS / (stem + '_boundary.png')
    ip = MASKS / (stem + '_interior.png')
    cp = MASKS / (stem + '_corners.png')
    vp = MASKS / (stem + '_valid.png')
    if all(p.exists() and p.stat().st_size > 0 for p in (bp, ip, cp, vp)):
        continue
    b, i_, c, v = heads_and_valid(r['tif'], r['mouzas'])
    cv2.imwrite(str(bp), b)
    cv2.imwrite(str(ip), i_)
    cv2.imwrite(str(cp), c)
    cv2.imwrite(str(vp), v)
print('masks ->', MASKS, '   (boundary, interior, corners, valid per tile)')


In [ ]:
# --- Cell 5: build patch lists (drop patches that don't overlap valid region) ---
PATCH  = CFG['dataset']['patch_size']
STRIDE = CFG['dataset']['stride']

def is_train_tile(r): return all(m in train_mouzas for m in r['mouzas'])
def is_val_tile(r):   return all(m in val_mouzas   for m in r['mouzas'])

train_patches, val_patches = [], []
for r in rows:
    target = train_patches if is_train_tile(r) else (val_patches if is_val_tile(r) else None)
    if target is None: continue
    with rasterio.open(r['tif']) as ds: H, W = ds.height, ds.width
    stem = Path(r['tif']).stem
    bp = MASKS / (stem + '_boundary.png')
    ip = MASKS / (stem + '_interior.png')
    cp = MASKS / (stem + '_corners.png')
    vp = MASKS / (stem + '_valid.png')
    for top in range(0, H - PATCH + 1, STRIDE):
        for left in range(0, W - PATCH + 1, STRIDE):
            target.append({'tif': str(r['tif']),
                           'boundary': str(bp), 'interior': str(ip),
                           'corners':  str(cp), 'valid':    str(vp),
                           'top': top, 'left': left})

if CFG['dataset']['drop_empty_patches']:
    print('Filtering patches with no valid coverage...')
    cache = {}
    def get(path):
        if path not in cache: cache[path] = cv2.imread(path, cv2.IMREAD_UNCHANGED)
        return cache[path]
    def has_valid(p):
        v = get(p['valid'])
        return v[p['top']:p['top']+PATCH, p['left']:p['left']+PATCH].any()
    train_patches = [p for p in train_patches if has_valid(p)]
    val_patches   = [p for p in val_patches   if has_valid(p)]

print(f'Train patches: {len(train_patches)}   Val patches: {len(val_patches)}')
assert train_patches, 'No train patches. Stage more mouzas (raise max_mouzas in config).'
assert val_patches,   'No val patches.'


In [ ]:
# --- Cell 6: copy tiles + 4 masks per tile to Colab's local SSD ---
import shutil, time
LOCAL = Path('/content/local_train')
LOCAL_TILES = LOCAL / 'tiles'; LOCAL_TILES.mkdir(parents=True, exist_ok=True)
LOCAL_MASKS = LOCAL / 'masks'; LOCAL_MASKS.mkdir(parents=True, exist_ok=True)

unique_tifs = sorted({Path(p['tif']) for p in train_patches + val_patches})
t0 = time.time()
for src in unique_tifs:
    dst = LOCAL_TILES / src.name
    if not dst.exists() or dst.stat().st_size != src.stat().st_size:
        print(f'  copying {src.name} ({src.stat().st_size/1e6:.0f} MB) ...')
        shutil.copy2(src, dst)
    for suffix in ('_boundary.png', '_interior.png', '_corners.png', '_valid.png'):
        m_src = MASKS / (src.stem + suffix)
        m_dst = LOCAL_MASKS / m_src.name
        if m_src.exists() and (not m_dst.exists() or m_dst.stat().st_size != m_src.stat().st_size):
            shutil.copy2(m_src, m_dst)

for p in train_patches + val_patches:
    p['tif']      = str(LOCAL_TILES / Path(p['tif']).name)
    for key in ('boundary', 'interior', 'corners', 'valid'):
        p[key] = str(LOCAL_MASKS / Path(p[key]).name)

local_gb = sum(f.stat().st_size for f in LOCAL_TILES.glob('*.tif')) / 1e9
print(f'\nLocal cache: {local_gb:.2f} GB in {time.time()-t0:.0f}s')


In [ ]:
# --- Cell 7: pre-extract ALL patches (img + 3 targets + valid) into RAM ---
# Memory cost rule of thumb: N_patches * (3 + 4) * PATCH^2 bytes (boundary, interior,
# corners, valid are each uint8 — corners is the gaussian heatmap rescaled to 0..255).
# 1856 patches @ 512^2 -> ~3.4 GB. Colab has 12+ GB RAM.

def extract_all(patches, name):
    n = len(patches)
    imgs       = np.empty((n, PATCH, PATCH, 3), dtype=np.uint8)
    boundaries = np.empty((n, PATCH, PATCH),    dtype=np.uint8)
    interiors  = np.empty((n, PATCH, PATCH),    dtype=np.uint8)
    corners    = np.empty((n, PATCH, PATCH),    dtype=np.uint8)
    valids     = np.empty((n, PATCH, PATCH),    dtype=np.uint8)
    by_tif = {}
    for i, p in enumerate(patches):
        by_tif.setdefault(p['tif'], []).append((i, p))
    for tif_path, items in tqdm(by_tif.items(), desc=f'extract {name}'):
        with rasterio.open(tif_path) as ds:
            tile = ds.read([1, 2, 3])
        tile = np.transpose(tile, (1, 2, 0))
        b_full  = cv2.imread(items[0][1]['boundary'], cv2.IMREAD_UNCHANGED)
        i_full  = cv2.imread(items[0][1]['interior'], cv2.IMREAD_UNCHANGED)
        c_full  = cv2.imread(items[0][1]['corners'],  cv2.IMREAD_UNCHANGED)
        v_full  = cv2.imread(items[0][1]['valid'],    cv2.IMREAD_UNCHANGED)
        for i, p in items:
            t, l = p['top'], p['left']
            imgs[i]       = tile[t:t+PATCH, l:l+PATCH]
            boundaries[i] = (b_full[t:t+PATCH, l:l+PATCH] > 0).astype(np.uint8)
            interiors[i]  = (i_full[t:t+PATCH, l:l+PATCH] > 0).astype(np.uint8)
            corners[i]    =  c_full[t:t+PATCH, l:l+PATCH]            # keep 0..255 (heatmap)
            valids[i]     = (v_full[t:t+PATCH, l:l+PATCH] > 0).astype(np.uint8)
        del tile, b_full, i_full, c_full, v_full
    return imgs, boundaries, interiors, corners, valids

print(f'Pre-extracting {len(train_patches)} train + {len(val_patches)} val patches...')
train_imgs, train_b, train_i, train_c, train_v = extract_all(train_patches, 'train')
val_imgs,   val_b,   val_i,   val_c,   val_v   = extract_all(val_patches,   'val')
total_gb = sum(a.nbytes for a in (train_imgs, train_b, train_i, train_c, train_v,
                                  val_imgs,   val_b,   val_i,   val_c,   val_v)) / 1e9
print(f'\nCached in RAM: train {train_imgs.shape}  val {val_imgs.shape}   = {total_gb:.2f} GB')
print(f'  train valid coverage: {(train_v > 0).mean()*100:.1f}% of pixels')
print(f'  val   valid coverage: {(val_v   > 0).mean()*100:.1f}% of pixels')
print(f'  train boundary positive: {(train_b > 0).mean()*100:.2f}%   interior positive: {(train_i > 0).mean()*100:.2f}%')


In [ ]:
# --- Cell 8: in-memory Dataset that yields (img, boundary, interior, corners, valid) ---
import torch
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

def build_aug(train):
    A_CFG = CFG['augment']
    if train:
        return A.Compose([
            A.HorizontalFlip(p=A_CFG['hflip_p']),
            A.VerticalFlip(p=A_CFG['vflip_p']),
            A.RandomRotate90(p=A_CFG['rot90_p']),
            A.RandomBrightnessContrast(brightness_limit=A_CFG['brightness'],
                                       contrast_limit=A_CFG['contrast'], p=0.5),
            A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
            ToTensorV2(),
        ], additional_targets={'interior': 'mask', 'corners': 'mask', 'valid': 'mask'})
    return A.Compose([
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ], additional_targets={'interior': 'mask', 'corners': 'mask', 'valid': 'mask'})

class InMemoryDataset(Dataset):
    """Yields (img, target_3xHxW, valid_1xHxW). target channels: boundary, interior, corners."""
    def __init__(self, imgs, b, i_, c, v, train):
        self.imgs, self.b, self.i, self.c, self.v = imgs, b, i_, c, v
        self.aug = build_aug(train)
    def __len__(self): return len(self.imgs)
    def __getitem__(self, idx):
        out = self.aug(image    = self.imgs[idx],
                       mask     = self.b[idx].astype(np.float32),                 # boundary 0/1
                       interior = self.i[idx].astype(np.float32),                 # interior 0/1
                       corners  = (self.c[idx].astype(np.float32) / 255.0),       # corners 0..1
                       valid    = self.v[idx].astype(np.float32))                 # valid 0/1
        target = torch.stack([out['mask'], out['interior'], out['corners']], dim=0)
        return out['image'], target, out['valid'].unsqueeze(0)

train_ds = InMemoryDataset(train_imgs, train_b, train_i, train_c, train_v, train=True)
val_ds   = InMemoryDataset(val_imgs,   val_b,   val_i,   val_c,   val_v,   train=False)
BS = CFG['train']['batch_size']
train_dl = DataLoader(train_ds, batch_size=BS, shuffle=True,  num_workers=0, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BS, shuffle=False, num_workers=0, pin_memory=True)
print(f'train batches: {len(train_dl)}   val batches: {len(val_dl)}')


In [ ]:
# --- Cell 9: multi-task UNet++ + valid-masked train loop ---
import segmentation_models_pytorch as smp
import torch.nn.functional as F
from torch.amp import autocast, GradScaler

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

M = CFG['model']
model = getattr(smp, M['arch'])(
    encoder_name=M['encoder'], encoder_weights=M['encoder_weights'],
    in_channels=M['in_channels'], classes=M['classes'],
).to(DEVICE)
print(f"model: {M['arch']} / {M['encoder']} / classes={M['classes']}")

L = CFG['train']['loss']
BCE_W, DICE_W = L['bce_weight'], L['dice_weight']
W_B, W_I, W_C = L['boundary_weight'], L['interior_weight'], L['corners_weight']

def masked_bce(logits, y, v):
    """Per-pixel BCE, masked to the valid coverage region."""
    per_pix = F.binary_cross_entropy_with_logits(logits, y, reduction='none')
    return (per_pix * v).sum() / v.sum().clamp(min=1.0)

def masked_dice(logits, y, v, eps=1.0):
    p  = torch.sigmoid(logits) * v
    yv = y * v
    inter = (p * yv).sum(dim=(2, 3))                      # (N, C)
    denom = p.sum(dim=(2, 3)) + yv.sum(dim=(2, 3))
    return (1 - (2 * inter + eps) / (denom + eps)).mean()

def masked_mse(pred_sigmoid, y, v):
    """For the corner heatmap (real-valued in [0,1]) we use masked MSE, not BCE."""
    return ((pred_sigmoid - y) ** 2 * v).sum() / v.sum().clamp(min=1.0)

def loss_fn(logits, target, v):
    """logits: (N, 3, H, W)  target: (N, 3, H, W)  v: (N, 1, H, W)
       Channel 0 boundary BCE+Dice. Channel 1 interior BCE+Dice. Channel 2 corners MSE+Dice."""
    # head 0: boundary
    b_logits = logits[:, 0:1]; b_y = target[:, 0:1]
    L_b = BCE_W * masked_bce(b_logits, b_y, v) + DICE_W * masked_dice(b_logits, b_y, v)
    # head 1: interior
    i_logits = logits[:, 1:2]; i_y = target[:, 1:2]
    L_i = BCE_W * masked_bce(i_logits, i_y, v) + DICE_W * masked_dice(i_logits, i_y, v)
    # head 2: corner heatmap — MSE on sigmoid + soft dice
    c_logits = logits[:, 2:3]; c_y = target[:, 2:3]
    c_sig = torch.sigmoid(c_logits)
    L_c = masked_mse(c_sig, c_y, v) + 0.5 * masked_dice(c_logits, (c_y > 0.2).float(), v)
    return W_B * L_b + W_I * L_i + W_C * L_c, {'b': L_b.item(), 'i': L_i.item(), 'c': L_c.item()}

def masked_iou(logits_1ch, y_1ch, v, thr=0.5, eps=1e-7):
    p  = (torch.sigmoid(logits_1ch) > thr).float() * v
    yv = (y_1ch > 0.5).float() * v
    inter = (p * yv).sum(dim=(1, 2, 3))
    union = ((p + yv) >= 1).float().sum(dim=(1, 2, 3))
    return ((inter + eps) / (union + eps)).mean().item()

opt = torch.optim.AdamW(model.parameters(), lr=CFG['train']['lr'], weight_decay=CFG['train']['weight_decay'])
EPOCHS = CFG['train']['epochs']
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
scaler = GradScaler('cuda', enabled=CFG['train']['amp'])

CKPT = STAGE / 'checkpoints'; CKPT.mkdir(exist_ok=True)
best_score = -1.0   # we early-pick on (boundary_iou + interior_iou) / 2

for epoch in range(1, EPOCHS + 1):
    t_ep = time.time()
    model.train(); tr_loss = 0.0; tr_parts = {'b': 0.0, 'i': 0.0, 'c': 0.0}
    for x, y, v in train_dl:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        v = v.to(DEVICE, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with autocast('cuda', enabled=CFG['train']['amp']):
            logits = model(x)
            loss, parts = loss_fn(logits, y, v)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        tr_loss += loss.item() * x.size(0)
        for k in parts: tr_parts[k] += parts[k] * x.size(0)
    tr_loss /= max(1, len(train_ds))
    for k in tr_parts: tr_parts[k] /= max(1, len(train_ds))

    model.eval(); v_loss, b_iou, i_iou, n = 0.0, 0.0, 0.0, 0
    with torch.no_grad():
        for x, y, v in val_dl:
            x, y, v = x.to(DEVICE), y.to(DEVICE), v.to(DEVICE)
            logits = model(x); loss, _ = loss_fn(logits, y, v)
            v_loss += loss.item() * x.size(0)
            b_iou  += masked_iou(logits[:, 0:1], y[:, 0:1], v) * x.size(0)
            i_iou  += masked_iou(logits[:, 1:2], y[:, 1:2], v) * x.size(0)
            n      += x.size(0)
    v_loss /= max(1, n); b_iou /= max(1, n); i_iou /= max(1, n)
    score = 0.5 * (b_iou + i_iou)
    sched.step()

    dt = time.time() - t_ep
    print(f'ep {epoch:02d}/{EPOCHS}  train={tr_loss:.4f} (b={tr_parts["b"]:.3f} i={tr_parts["i"]:.3f} c={tr_parts["c"]:.3f})  '
          f'val={v_loss:.4f}  iou_b={b_iou:.3f}  iou_i={i_iou:.3f}  score={score:.3f}  '
          f'lr={opt.param_groups[0]["lr"]:.2e}  {dt:.0f}s')
    if score > best_score:
        best_score = score
        torch.save({'model': model.state_dict(), 'epoch': epoch,
                    'val_iou': score, 'val_iou_boundary': b_iou, 'val_iou_interior': i_iou,
                    'cfg': CFG}, CKPT / 'best.pt')
        print(f'  saved best (score={score:.4f}  b={b_iou:.3f}  i={i_iou:.3f})')

print(f'\nDone. best (boundary+interior)/2 = {best_score:.4f}')


In [ ]:
# --- Cell 10: qualitative eval — all 3 heads + fitted polygons on a val tile ---
import matplotlib.pyplot as plt, sys
sys.path.insert(0, str(Path('src').resolve()))
from inference import extract_polygons, draw_polygons     # vectoriser + drawer from src/

val_tile_rows = [r for r in rows if all(m in val_mouzas for m in r['mouzas'])]
assert val_tile_rows, 'No val tile found.'
vr = val_tile_rows[0]
vr_tif = LOCAL_TILES / Path(vr['tif']).name if (LOCAL_TILES / Path(vr['tif']).name).exists() else vr['tif']
stem = Path(vr['tif']).stem
def _pick(name):
    p_local = LOCAL_MASKS / (stem + name)
    return p_local if p_local.exists() else (MASKS / (stem + name))
vr_b, vr_i, vr_c, vr_v = (_pick(s) for s in ('_boundary.png', '_interior.png', '_corners.png', '_valid.png'))
print('Visualising:', Path(vr_tif).name, '  mouzas:', vr['mouzas'])

CROP = 1024
with rasterio.open(vr_tif) as ds:
    H, W = ds.height, ds.width
    top, left = H // 2 - CROP // 2, W // 2 - CROP // 2
    crop = ds.read([1, 2, 3], window=rasterio.windows.Window(left, top, CROP, CROP))
crop = np.transpose(crop, (1, 2, 0))

# Sliding-window inference over the crop, 3 heads.
model.eval()
norm = A.Compose([A.Normalize(), ToTensorV2()])
pred = np.zeros((3, CROP, CROP), dtype=np.float32)
cnt  = np.zeros((CROP, CROP),    dtype=np.float32)
with torch.no_grad():
    for top2 in range(0, CROP - PATCH + 1, STRIDE):
        for left2 in range(0, CROP - PATCH + 1, STRIDE):
            sub = crop[top2:top2+PATCH, left2:left2+PATCH]
            x = norm(image=sub)['image'].unsqueeze(0).to(DEVICE)
            p = torch.sigmoid(model(x))[0].cpu().numpy()      # (3, h, w)
            pred[:, top2:top2+PATCH, left2:left2+PATCH] += p
            cnt[top2:top2+PATCH, left2:left2+PATCH]     += 1
pred = pred / np.maximum(cnt, 1)
heatmaps = {'boundary': pred[0], 'interior': pred[1], 'corners': pred[2]}

# Vectorise to polygons using the v2.0 extractor.
POLY_CFG = CFG['polygons']
polys = extract_polygons(
    heatmaps,
    threshold=POLY_CFG['interior_threshold'],
    close_kernel=POLY_CFG['close_kernel'],
    min_area_px=POLY_CFG['min_area_px'],
    approx_eps_frac=POLY_CFG['approx_eps_frac'],
    regularise_to_rect=True,
    rect_iou_threshold=POLY_CFG['rect_iou_threshold'],
    trapezoid_iou_threshold=POLY_CFG['trapezoid_iou_threshold'],
    quad_iou_threshold=POLY_CFG['quad_iou_threshold'],
    corner_threshold=POLY_CFG['corner_threshold'],
    corner_search_band_px=POLY_CFG['corner_search_band_px'],
    source='interior',
)
print(f'extracted {len(polys)} polygons (rect / trapezoid / quad / general)')

# GT crops + masked vs unmasked boundary IoU (sanity check that the boundary head is learning).
gt_b = (cv2.imread(str(vr_b), cv2.IMREAD_UNCHANGED) > 0).astype(np.uint8)
gt_i = (cv2.imread(str(vr_i), cv2.IMREAD_UNCHANGED) > 0).astype(np.uint8)
val_full = (cv2.imread(str(vr_v), cv2.IMREAD_UNCHANGED) > 0).astype(np.uint8)
gt_b_c = gt_b[top:top+CROP, left:left+CROP]
gt_i_c = gt_i[top:top+CROP, left:left+CROP]
val_c  = val_full[top:top+CROP, left:left+CROP]

def crop_iou(p, g, v=None):
    p_b = (p > 0.5); g_b = (g > 0)
    if v is not None:
        p_b = p_b & (v > 0); g_b = g_b & (v > 0)
    inter = int((p_b & g_b).sum()); union = int((p_b | g_b).sum())
    return inter / max(1, union)

iou_b_u = crop_iou(pred[0], gt_b_c);              iou_b_m = crop_iou(pred[0], gt_b_c, val_c)
iou_i_u = crop_iou(pred[1], gt_i_c);              iou_i_m = crop_iou(pred[1], gt_i_c, val_c)
print(f'crop IoU boundary unmasked={iou_b_u:.3f}  masked={iou_b_m:.3f}')
print(f'crop IoU interior unmasked={iou_i_u:.3f}  masked={iou_i_m:.3f}')

fig, ax = plt.subplots(2, 3, figsize=(20, 14))
ax[0, 0].imshow(crop);                                                        ax[0, 0].set_title('RGB')
ax[0, 1].imshow(crop); ax[0, 1].imshow(pred[0], cmap='Reds',   alpha=0.55); ax[0, 1].set_title(f'Boundary head (IoU masked {iou_b_m:.2f})')
ax[0, 2].imshow(crop); ax[0, 2].imshow(pred[1], cmap='Greens', alpha=0.45); ax[0, 2].set_title(f'Interior head (IoU masked {iou_i_m:.2f})')
ax[1, 0].imshow(crop); ax[1, 0].imshow(pred[2], cmap='magma',  alpha=0.60); ax[1, 0].set_title('Corner heatmap')
ax[1, 1].imshow(draw_polygons(crop, polys, color=(255, 215, 0), thickness=2, fill_alpha=0.0));  ax[1, 1].set_title(f'Fitted polygons ({len(polys)})')
ax[1, 2].imshow(crop); ax[1, 2].imshow(val_c, cmap='Blues', alpha=0.35);    ax[1, 2].set_title('Valid coverage (GT region)')
for a in ax.ravel(): a.axis('off')
plt.tight_layout(); plt.show()
